# Data Validation with pandera

Every model and pipeline makes **assumptions** about its input: column names, types, ranges, allowed categories. When new data breaks an assumption, the model often keeps running and produces wrong predictions silently. **pandera** writes these assumptions as a schema and checks every DataFrame against it.

**After this lesson you can:**

1. Define schemas with column types, nullability, value checks and allowed categories.
2. Write custom and cross-column checks.
3. Validate lazily and turn the failures into a report.
4. Use schemas as gates in a pipeline: at ingestion, before training, before prediction.

**Data:** Adult census income (training data and the official test file).

In [1]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

sys.path.insert(0, str(Path.cwd().parent / "tools"))
import datasets                                   # course helper: downloads pinned public datasets

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})
RNG = 42

import pandera.pandas as pa
from pandera.errors import SchemaErrors

adult = datasets.adult()
test = datasets.adult("test")
print("pandera", pa.__version__ if hasattr(pa, "__version__") else __import__("pandera").__version__)

pandera 0.36.0


## 1. A Schema for the Training Data

The schema states what the pipeline of Lesson 10 expects. The allowed categories are taken from the training data, so new categories are detected.

In [2]:
CATEGORIES = {c: sorted(adult[c].dropna().unique()) for c in ["workclass", "marital_status", "relationship", "race", "sex"]}

schema = pa.DataFrameSchema(
    {
        "age": pa.Column(int, pa.Check.in_range(17, 90)),
        "workclass": pa.Column(str, pa.Check.isin(CATEGORIES["workclass"]), nullable=True),
        "education_num": pa.Column(int, pa.Check.in_range(1, 16)),
        "marital_status": pa.Column(str, pa.Check.isin(CATEGORIES["marital_status"])),
        "relationship": pa.Column(str, pa.Check.isin(CATEGORIES["relationship"])),
        "race": pa.Column(str, pa.Check.isin(CATEGORIES["race"])),
        "sex": pa.Column(str, pa.Check.isin(CATEGORIES["sex"])),
        "capital_gain": pa.Column(int, [pa.Check.ge(0), pa.Check.le(99999)]),
        "capital_loss": pa.Column(int, pa.Check.ge(0)),
        "hours_per_week": pa.Column(int, pa.Check.in_range(1, 99)),
        "income": pa.Column(str, pa.Check.isin(["<=50K", ">50K"])),
    },
    checks=[
        pa.Check(lambda df: ~((df["relationship"] == "Husband") & (df["sex"] == "Female")), name="husband_is_male",
                 element_wise=False),
        pa.Check(lambda df: len(df) > 1000, name="enough_rows", element_wise=False),
    ],
    strict=False,                        # extra columns are allowed
    coerce=False,
)
try:
    schema.validate(adult, lazy=True)
except SchemaErrors as err:
    print("even the TRAINING data breaks a rule:")
    print(err.failure_cases[["check", "index"]].drop_duplicates().to_string(index=False))
odd = adult.loc[(adult["relationship"] == "Husband") & (adult["sex"] == "Female")]
print(odd[["age", "relationship", "sex", "marital_status"]])
adult = adult.drop(index=odd.index)                          # documented fix: inconsistent record removed
validated = schema.validate(adult)
print("after removing the inconsistent record, the training data passes:", validated.shape)

even the TRAINING data breaks a rule:
          check  index
husband_is_male   7109
      age relationship     sex      marital_status
7109   34      Husband  Female  Married-civ-spouse
after removing the inconsistent record, the training data passes: (32560, 15)


The census file contains one person recorded as a "Husband" and "Female" at the same time: a data-entry error that no model would notice, found by a one-line rule.

## 2. Does the Official Test File Pass?

In [3]:
try:
    schema.validate(test, lazy=True)
    print("test file passes")
except SchemaErrors as err:
    report = err.failure_cases
    print(f"{len(report)} failing values")
    summary = report.groupby(["column", "check"], dropna=False).size().rename("failures").reset_index()
    print(summary.to_string(index=False))

test file passes


In [4]:
if 'report' in dir() and len(report):
    print(report.head(8)[["column", "check", "failure_case", "index"]].to_string(index=False))

The official test file passes: it comes from the same census, with the same categories and ranges. If it failed, `lazy=True` would collect **all** failures instead of stopping at the first one, giving a complete report for the data provider (Section 4 shows such a report for a broken batch). Some failures in such reports are real data problems; others show that the schema is too strict and must be updated. A schema is a living contract.

## 3. Class-Based Schemas (DataFrameModel)

pandera also supports a class syntax with type annotations, which reads like a data contract and works with type checkers.

In [5]:
from pandera.typing import Series

class Person(pa.DataFrameModel):
    age: Series[int] = pa.Field(ge=17, le=90)
    hours_per_week: Series[int] = pa.Field(ge=1, le=99)
    capital_gain: Series[int] = pa.Field(ge=0)
    sex: Series[str] = pa.Field(isin=["Female", "Male"])

    @pa.dataframe_check
    def working_age_hours(cls, df: pd.DataFrame) -> Series[bool]:
        """People above 80 working more than 60 hours a week are suspicious."""
        return ~((df["age"] > 80) & (df["hours_per_week"] > 60))

    class Config:
        strict = False

try:
    Person.validate(adult, lazy=True)
    print("Person schema: all rows pass")
except SchemaErrors as err:
    print(err.failure_cases[["check", "failure_case"]].value_counts().head())

check              failure_case 
working_age_hours  0                5
                   Private          3
                   White            3
                   Male             3
                   United-States    3
Name: count, dtype: int64


## 4. Schemas as Pipeline Gates

The cell simulates a broken delivery (wrong types, out-of-range values, a new category, a renamed column) and shows a **gate** function that rejects the batch before it reaches the model, with a readable report.

In [6]:
batch = adult.drop(columns="income").sample(200, random_state=RNG).copy()
batch.loc[batch.index[:3], "age"] = [-1, 150, 0]
batch.loc[batch.index[3:6], "sex"] = "female"                        # wrong case from a new source system
batch.loc[batch.index[6:9], "hours_per_week"] = 0
batch = batch.rename(columns={"marital_status": "maritalStatus"})    # a renamed column

prediction_schema = schema.remove_columns(["income"])

def gate(df: pd.DataFrame) -> tuple[bool, pd.DataFrame]:
    try:
        prediction_schema.validate(df, lazy=True)
        return True, pd.DataFrame()
    except SchemaErrors as err:
        fc = err.failure_cases
        return False, fc.groupby(["column", "check"], dropna=False)["failure_case"].agg(lambda s: list(s.unique())[:4]).reset_index()

ok, problems = gate(batch)
print("batch accepted:", ok)
problems

batch accepted: False


,column,check,failure_case
0,age,"in_range(17, 90)","[-1, 150, 0]"
1,hours_per_week,"in_range(1, 99)",[0]
2,sex,"isin(['Female', 'Male'])",[female]
3,NaN,column_in_dataframe,[marital_status]
4,NaN,enough_rows,[False]


| Where to validate | What it protects |
| :--- | :--- |
| ingestion (raw data arrives) | the data warehouse from bad deliveries |
| before training | the model from learning on broken data |
| before prediction (online or batch) | users from predictions on impossible input |
| after prediction | downstream systems from impossible outputs (for example probabilities outside [0, 1]) |

Other tools follow the same idea: Great Expectations (expectation suites and data docs), dbt tests (M11), Pydantic (single records in APIs, M01/M30), and data contracts between teams (M12).

*Insight: Write the assumptions of your pipeline as a schema and check them where data enters and leaves your system. Validate lazily, report every failure, and treat schema changes like code changes: reviewed and versioned.*

## Real-World Use Case or Analogy

A schema is like the **customs check at a border**: every shipment is checked against a list of rules (documents, weight, forbidden items). A shipment that fails is stopped with a written report, instead of being discovered later in a shop.

Data teams commonly find that most production incidents of ML systems are data problems (schema changes, new categories, unit changes), not model problems. Validation gates catch them before customers do.